# 16. Optimization — Collection / Resource Allocation

## Business Case

A bank has a portfolio of overdue customers.

The collection team has limited resources:

- limited call-center capacity,
- limited field-agent capacity,
- limited digital campaign budget.

The business question is:

> **How should limited collection resources be allocated to maximize expected recovery?**

This is an **Optimization** problem.

The objective is not simply to predict who will pay.

Instead:

```text
Predictions
    ↓
Expected recovery
    ↓
Costs + constraints
    ↓
Optimization
    ↓
Best allocation
```

This notebook uses synthetic banking data for educational purposes.

## 1. Prediction vs Optimization

### Prediction

Question:

> How much might we recover from this customer?

Output:

```text
Expected Recovery = Rp 2,500,000
```

### Optimization

Question:

> Given only 650 call-center slots and 300 field-agent slots, which customers should receive those resources?

Optimization considers:

```text
Expected benefit
+
Resource cost
+
Business constraints
```

The distinction is important:

> **Machine Learning predicts. Optimization decides how to allocate resources under constraints.**

## 2. Banking Collection Example

Suppose:

```text
10,000 overdue customers

Resources:
650 call slots
300 field-agent visits
Rp 7.5 billion digital budget
```

A naive strategy may prioritize the largest overdue balance.

But that can be suboptimal.

For example:

```text
Customer A
Overdue = Rp 100m
Expected recovery = Rp 2m

Customer B
Overdue = Rp 20m
Expected recovery = Rp 10m
```

Customer B may create more expected recovery despite having a smaller balance.

Optimization makes the trade-off explicit.

## 3. Decision Variables

For each customer, define:

```text
x_i,c = 1
```

if customer `i` receives collection channel `c`.

Otherwise:

```text
x_i,c = 0
```

Example:

```text
x(Customer 001, Field) = 1
x(Customer 001, SMS)   = 0
```

The optimizer chooses these variables.

## 4. Objective Function

A simple objective is:

```text
Maximize

Σ Expected Recovery(i,c) × x(i,c)
-
Σ Cost(i,c) × x(i,c)
```

or, if maximizing gross recovery:

```text
Maximize

Σ Expected Recovery(i,c) × x(i,c)
```

subject to resource constraints.

This is a form of **Linear / Integer Programming** depending on the formulation.

## 5. Constraints

Typical banking constraints include:

### Capacity

```text
Call contacts ≤ available agents × daily capacity
```

### Field visits

```text
Field visits ≤ field-agent capacity
```

### Budget

```text
Campaign cost ≤ campaign budget
```

### Customer assignment

```text
One customer → at most one primary action
```

### Policy

Certain customers may require:

```text
Field visit
```

or may be restricted from certain channels.

Optimization is where these business rules become mathematically explicit.

## 6. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import milp, LinearConstraint, Bounds

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

df=pd.read_csv("bank_collection_resource_allocation_sample.csv")
capacity=pd.read_csv("collection_resource_capacity.csv")

print("Customers:",len(df))
display(df.head())
display(capacity)

## 7. Dataset Dictionary

In [ ]:
dictionary=pd.DataFrame({
    "Column":[
        "Customer_ID","Risk_Score","Outstanding_Balance",
        "Overdue_Days","Overdue_Amount","Segment",
        "Current_Channel","Estimated_Recovery_Rate",
        "Expected_Recovery","Contact_Cost","Recovery_per_Cost"
    ],
    "Meaning":[
        "Customer identifier",
        "Synthetic risk score",
        "Outstanding balance",
        "Days past due",
        "Estimated overdue amount",
        "Customer segment",
        "Example current channel",
        "Estimated recovery probability/rate",
        "Expected recovery amount",
        "Cost of current action",
        "Expected recovery divided by cost"
    ]
})
display(dictionary)

## 8. Data Quality

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("Missing"))

print("Duplicate customers:",df["Customer_ID"].duplicated().sum())

display(
    df.describe(
        include="all"
    ).T.head(20)
)

## 9. Portfolio Overview

In [ ]:
portfolio=pd.DataFrame({
    "Metric":[
        "Customers",
        "Total Outstanding",
        "Total Overdue",
        "Average Overdue",
        "Total Expected Recovery"
    ],
    "Value":[
        len(df),
        df["Outstanding_Balance"].sum(),
        df["Overdue_Amount"].sum(),
        df["Overdue_Amount"].mean(),
        df["Expected_Recovery"].sum()
    ]
})

display(portfolio)

## 10. Expected Recovery Distribution

In [ ]:
plt.figure(figsize=(9,5))

sns.histplot(
    df["Expected_Recovery"],
    bins=40
)

plt.title("Expected Recovery Distribution")
plt.xlabel("Expected Recovery")
plt.show()

## 11. Simple Heuristic Baseline

Before optimization, create a simple rule:

> Prioritize customers with the highest overdue amount.

This is a realistic baseline.

Then compare it with an optimization strategy.

The purpose is not to assume the heuristic is bad, but to establish a measurable benchmark.

In [ ]:
CALL_CAPACITY=650
FIELD_CAPACITY=300
DIGITAL_BUDGET=7_500_000

baseline=df.sort_values(
    "Overdue_Amount",
    ascending=False
).head(CALL_CAPACITY).copy()

baseline_recovery=baseline["Expected_Recovery"].sum()

print("Baseline selected:",len(baseline))
print("Expected recovery:",round(baseline_recovery))

## 12. Greedy Value-per-Cost Baseline

Another heuristic is:

```text
Expected Recovery
-----------------
Cost
```

This prioritizes customers by expected recovery per unit of resource cost.

It is simple and often useful, but it does not automatically handle all constraints or interactions.

In [ ]:
greedy=df.sort_values(
    "Recovery_per_Cost",
    ascending=False
).head(CALL_CAPACITY).copy()

greedy_recovery=greedy["Expected_Recovery"].sum()
greedy_cost=greedy["Contact_Cost"].sum()

print("Greedy selected:",len(greedy))
print("Expected recovery:",round(greedy_recovery))
print("Cost:",round(greedy_cost))

## 13. Why Optimization?

Imagine:

```text
Customer A → high recovery, expensive field visit
Customer B → moderate recovery, cheap SMS
Customer C → high recovery, requires field visit
```

A simple ranking may allocate too many expensive resources.

Optimization can decide:

```text
Who?
Which channel?
How many?
Under what budget?
```

while simultaneously respecting constraints.

## 14. Create a Multi-Channel Decision Problem

For demonstration, each customer can receive one of:

```text
0 = No action
1 = SMS
2 = WhatsApp
3 = Call Center
4 = Field Agent
```

Each action has:

- expected recovery,
- cost,
- resource consumption.

The optimization chooses the action with maximum total expected net value subject to capacity constraints.

In [ ]:
channels=[
    "SMS",
    "WhatsApp",
    "Call Center",
    "Field Agent"
]

costs={
    "SMS":3000,
    "WhatsApp":5000,
    "Call Center":25000,
    "Field Agent":180000
}

base_rates={
    "SMS":.20,
    "WhatsApp":.27,
    "Call Center":.42,
    "Field Agent":.58
}

# Build an action table
actions=[]

for i,row in df.iterrows():
    for ch in channels:
        rate=(
            base_rates[ch]
            * (.65+.7*(1-row["Risk_Score"]))
            * (.7+.3*min(row["Overdue_Days"],90)/90)
        )
        rate=np.clip(rate,.03,.85)

        expected=row["Overdue_Amount"]*rate

        actions.append({
            "Customer_Index":i,
            "Customer_ID":row["Customer_ID"],
            "Channel":ch,
            "Expected_Recovery":expected,
            "Cost":costs[ch]
        })

actions=pd.DataFrame(actions)
actions["Net_Value"]=(
    actions["Expected_Recovery"]-
    actions["Cost"]
)

display(actions.head(10))

## 15. Optimization Formulation

Decision variable:

```text
x(i,c) ∈ {0,1}
```

Objective:

```text
Maximize
Σ NetValue(i,c) × x(i,c)
```

Subject to:

### At most one action per customer

```text
Σ_c x(i,c) ≤ 1
```

### Call capacity

```text
Σ_i x(i,Call) ≤ 650
```

### Field capacity

```text
Σ_i x(i,Field) ≤ 300
```

### Digital budget

```text
Σ_i,c DigitalCost(i,c) × x(i,c)
≤ Rp 7.5 billion
```

This is a **Mixed Integer Linear Programming** style formulation.

## 16. Solve the Resource Allocation Problem

In [ ]:
# Build variables for all customer-channel pairs
n_vars=len(actions)

c=-actions["Net_Value"].to_numpy()  # scipy minimizes

integrality=np.ones(n_vars)

lower=np.zeros(n_vars)
upper=np.ones(n_vars)

constraints=[]
lb=[]
ub=[]

# One action at most per customer
for idx in df.index:
    ids=actions.index[actions["Customer_Index"]==idx].to_numpy()
    row=np.zeros(n_vars)
    row[ids]=1
    constraints.append(row)
    lb.append(0)
    ub.append(1)

# Call capacity
row=np.zeros(n_vars)
row[actions["Channel"].eq("Call Center")]=1
constraints.append(row)
lb.append(0)
ub.append(CALL_CAPACITY)

# Field capacity
row=np.zeros(n_vars)
row[actions["Channel"].eq("Field Agent")]=1
constraints.append(row)
lb.append(0)
ub.append(FIELD_CAPACITY)

# Digital budget applies to SMS + WhatsApp
row=np.zeros(n_vars)
digital_mask=actions["Channel"].isin(["SMS","WhatsApp"])
row[digital_mask]=actions.loc[digital_mask,"Cost"]
constraints.append(row)
lb.append(0)
ub.append(DIGITAL_BUDGET)

A=np.vstack(constraints)

result=milp(
    c=c,
    integrality=integrality,
    bounds=Bounds(lower,upper),
    constraints=LinearConstraint(
        A,
        np.array(lb),
        np.array(ub)
    ),
    options={"time_limit":30}
)

print("Optimization success:",result.success)
print("Message:",result.message)

## 17. Extract Optimal Allocation

In [ ]:
actions["Selected"]=(
    result.x>0.5
).astype(int)

selected=actions[actions["Selected"]==1].copy()

print("Selected actions:",len(selected))

display(
    selected[
        [
            "Customer_ID",
            "Channel",
            "Expected_Recovery",
            "Cost",
            "Net_Value"
        ]
    ].head(20)
)

## 18. Allocation by Channel

In [ ]:
allocation=(
    selected.groupby("Channel")
    .agg(
        Customers=("Customer_ID","count"),
        Expected_Recovery=("Expected_Recovery","sum"),
        Cost=("Cost","sum"),
        Net_Value=("Net_Value","sum")
    )
    .reset_index()
)

display(allocation.round(0))

In [ ]:
plt.figure(figsize=(9,5))

sns.barplot(
    data=allocation,
    x="Channel",
    y="Customers"
)

plt.title("Optimized Customer Allocation")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()

## 19. Compare Strategies

We compare:

### Strategy 1 — Largest overdue balance

```text
Prioritize biggest debt
```

### Strategy 2 — Greedy recovery/cost

```text
Prioritize highest value per cost
```

### Strategy 3 — Optimization

```text
Maximize total net expected recovery
subject to constraints
```

This gives management a concrete business comparison.

In [ ]:
comparison=pd.DataFrame({
    "Strategy":[
        "Largest Overdue",
        "Greedy Recovery/Cost",
        "Optimization"
    ],
    "Expected_Recovery":[
        baseline["Expected_Recovery"].sum(),
        greedy["Expected_Recovery"].sum(),
        selected["Expected_Recovery"].sum()
    ],
    "Cost":[
        baseline["Contact_Cost"].sum(),
        greedy["Contact_Cost"].sum(),
        selected["Cost"].sum()
    ],
    "Net_Value":[
        (baseline["Expected_Recovery"]-baseline["Contact_Cost"]).sum(),
        (greedy["Expected_Recovery"]-greedy["Contact_Cost"]).sum(),
        selected["Net_Value"].sum()
    ]
})

display(comparison.round(0))

## 20. Resource Utilization

Optimization is not only about total recovery.

Management should also see whether resources are being fully utilized.

```text
Capacity
vs
Used
```

This supports operational planning.

In [ ]:
used_call=(selected["Channel"]=="Call Center").sum()
used_field=(selected["Channel"]=="Field Agent").sum()
used_digital=selected.loc[
    selected["Channel"].isin(["SMS","WhatsApp"]),
    "Cost"
].sum()

utilization=pd.DataFrame({
    "Resource":[
        "Call Center",
        "Field Agent",
        "Digital Budget"
    ],
    "Capacity":[
        CALL_CAPACITY,
        FIELD_CAPACITY,
        DIGITAL_BUDGET
    ],
    "Used":[
        used_call,
        used_field,
        used_digital
    ]
})

utilization["Utilization"]=(
    utilization["Used"]/
    utilization["Capacity"]
)

display(utilization.round(3))

## 21. Customer-Level Decision Output

The final output should be operational.

Example:

```text
Customer ID
↓
Recommended Action
↓
Expected Recovery
↓
Cost
↓
Net Expected Value
```

This can be delivered to:

- collection work queues,
- CRM,
- branch dashboards,
- call-center systems,
- field-agent applications.

In [ ]:
decision_output=selected[[
    "Customer_ID",
    "Channel",
    "Expected_Recovery",
    "Cost",
    "Net_Value"
]].sort_values(
    "Net_Value",
    ascending=False
)

display(decision_output.head(30))

## 22. Optimization Is Different from Ranking

Ranking:

```text
Customer
   ↓
Score
   ↓
Sort
```

Optimization:

```text
Customers
+
Actions
+
Costs
+
Capacity
+
Business Rules
        ↓
Optimal Allocation
```

A ranking model can be one input into an optimization problem.

But optimization explicitly handles constraints.

## 23. Example Business Constraints

Real banking collection optimization can include:

### Capacity

- collector availability,
- call center seats,
- field agents,
- branch workload.

### Budget

- contact cost,
- incentive budget,
- collection agency fees.

### Customer rules

- customer segment,
- geography,
- regulatory restrictions,
- contact frequency limits.

### Operational rules

- maximum calls per customer,
- minimum time between contacts,
- field visit coverage.

### Risk

- expected recovery,
- probability of repayment,
- customer affordability.

These can be translated into optimization constraints.

## 24. Multi-Objective Optimization

Sometimes the objective is not only recovery.

For example:

```text
Maximize:

Recovery
-
Collection Cost
-
Customer Friction
-
Operational Risk
```

Conceptually:

```text
Score =
w1 × Expected Recovery
-
w2 × Cost
-
w3 × Contact Friction
-
w4 × Risk
```

The weights should be defined by business stakeholders and validated empirically.

## 25. Integer Programming

Some decisions are binary:

```text
Contact customer? 0 / 1
```

Some are integer:

```text
Number of agents = 20
Number of visits = 300
```

Some can be continuous:

```text
Budget allocation = Rp 2.5 billion
```

The mathematical formulation determines which optimization technique is appropriate.

## 26. What-If Scenario Analysis

Optimization becomes especially useful when management asks:

> "What happens if we add 100 field-agent slots?"

or:

> "What if collection budget is reduced by 20%?"

or:

> "What if expected recovery rates decrease?"

The model can be rerun under alternative constraints.

In [ ]:
scenarios=[]

for field_capacity in [150,300,450,600]:
    scenarios.append({
        "Field_Capacity":field_capacity,
        "Current_Base_Optimization":"Re-run optimizer with this capacity"
    })

display(pd.DataFrame(scenarios))

## 27. Shadow Price / Marginal Value

Optimization can answer another important management question:

> **What is the value of one additional unit of resource?**

For example:

```text
+1 field-agent visit
        ↓
+Rp X expected net recovery
```

This helps management decide whether adding capacity is economically justified.

In formal optimization this relates to **dual values / shadow prices**.

## 28. Production Architecture

```text
Customer / Loan Data
        ↓
Feature Engineering
        ↓
Recovery Prediction
        ↓
Expected Recovery
        ↓
Optimization Engine
        ↓
Resource Allocation
        ↓
Collection CRM
        ↓
Actual Recovery
        ↓
Monitoring
        ↓
Model + Optimization Update
```

Prediction and optimization are separate layers:

```text
ML Layer
"What is likely to happen?"

Optimization Layer
"What should we do?"


## 29. Monitoring

Monitor both prediction and optimization.

### Prediction

- recovery calibration,
- MAE / RMSE,
- probability calibration,
- drift.

### Optimization

- expected vs actual recovery,
- resource utilization,
- cost per recovery,
- constraint violations,
- action distribution.

### Business

- recovery rate,
- recovered amount,
- collection cost,
- net recovery,
- customer complaints.

## 30. Governance

Collection optimization is a high-impact banking application.

Production systems should include:

- policy constraints,
- human oversight,
- auditability,
- explainable allocation logic,
- customer treatment controls,
- data privacy,
- fairness monitoring,
- regulatory compliance.

Optimization should not automatically override established collection policies or customer-protection requirements.

## 31. Common Mistakes

1. Optimizing predicted values without validating prediction quality.
2. Ignoring resource constraints.
3. Using only overdue balance as the objective.
4. Ignoring collection cost.
5. Mixing prediction and decision logic.
6. Allowing impossible allocations.
7. Ignoring operational capacity.
8. Ignoring customer contact limits.
9. Treating model estimates as guaranteed recovery.
10. Not performing scenario analysis.
11. Not monitoring actual vs expected recovery.
12. Optimizing short-term recovery without considering customer and policy constraints.

## 32. Final Executive Summary

### Business Question

> **How should the bank allocate limited collection resources to maximize expected recovery?**

### End-to-End

```text
Customer Data
      ↓
Expected Recovery
      ↓
Cost / Benefit
      ↓
Resource Constraints
      ↓
Optimization
      ↓
Recommended Action
      ↓
Collection Execution
      ↓
Actual Recovery
      ↓
Monitoring
```

### Key distinction

```text
Prediction
→ What is likely to happen?

Optimization
→ What should we do given constraints?
```

For banking collection:

> **Optimization converts predictive insights into an actionable resource-allocation decision.**

This framework can be extended to:

- collection agent allocation,
- branch staffing,
- RM assignment,
- call-center capacity,
- marketing budget allocation,
- credit-limit allocation,
- loan approval resource planning,
- ATM cash replenishment,
- liquidity allocation.